# Modélisation — Prédiction du prix au m² des appartements parisiens

Ce notebook entraîne et compare trois modèles de régression sur le dataset DVF enrichi :
1. **Régression Linéaire** — baseline interprétable
2. **Random Forest** — ensemble robuste, non-linéaire
3. **Gradient Boosting** — gradient boosting, état de l'art sur données tabulaires

**Métriques d'évaluation :** MAE (€/m²), RMSE (€/m²), R²

## 1. Imports et configuration

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

sys.path.append("../src")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

from data import load_dataset_split
from metrics import compute_metrics

# Style des graphiques
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

print("✅ Imports OK")

## 2. Chargement des données et feature engineering

In [ ]:
X_train, X_test, y_train, y_test = load_dataset_split()
X_train = X_train.copy()
X_test  = X_test.copy()

# ── Feature engineering supplémentaire ────────────────────────────────────────

# 1. Log de la surface (relation non-linéaire entre surface et prix)
if "log_surface" not in X_train.columns:
    X_train["log_surface"] = np.log1p(X_train["surface_reelle_bati"])
    X_test["log_surface"]  = np.log1p(X_test["surface_reelle_bati"])

# 2. Surface par pièce (proxy du standing du bien)
if "surface_par_piece" not in X_train.columns:
    X_train["surface_par_piece"] = X_train["surface_reelle_bati"] / X_train["nombre_pieces_principales"].clip(lower=1)
    X_test["surface_par_piece"]  = X_test["surface_reelle_bati"]  / X_test["nombre_pieces_principales"].clip(lower=1)

# 3. Target encoding arrondissement (calculé sur train uniquement → pas de fuite)
arr_mean = y_train.groupby(X_train["arrondissement"]).mean()
X_train["arrondissement_prix_moyen"] = X_train["arrondissement"].map(arr_mean)
X_test["arrondissement_prix_moyen"]  = X_test["arrondissement"].map(arr_mean)

# Cibles log-transformées (utilisées pour les modèles v2)
y_train_log = np.log(y_train)
y_test_log  = np.log(y_test)

print(f"X_train : {X_train.shape[0]:,} lignes × {X_train.shape[1]} features")
print(f"Prix médian train : {y_train.median():,.0f} €/m²")
X_train.head(3)

## 2b. Enrichissement avec les données de stationnement

Le dataset **"Stationnement sur voie publique"** (OpenData Paris, 65 663 emplacements) est chargé et filtré sur les places voiture uniquement (PAYANT MIXTE, PAYANT ROTATIF, GRATUIT → 24 874 places).

Méthode : BallTree haversine (même approche que exploration_data.ipynb).

**Features créées :**
- `distance_parking_plus_proche` : distance en mètres à la place voiture la plus proche
- `nb_parkings_500m` : nombre de places voiture dans un rayon de 500m

In [ ]:
import json
from sklearn.neighbors import BallTree

EARTH_RADIUS_M = 6_371_000

# ── Chargement et filtrage du dataset stationnement ───────────────────────────
with open("../data/external/stationnement-voie-publique-emplacements.json") as f:
    parking_data = json.load(f)

parking_df = pd.DataFrame(parking_data)

# Filtrage : uniquement les places voiture (pas vélos, motos, livraison...)
parking_voiture = parking_df[parking_df["regpri"].isin(["PAYANT MIXTE", "PAYANT ROTATIF", "GRATUIT"])].copy()

# Extraction des coordonnées GPS
parking_voiture["lat"] = parking_voiture["geo_point_2d"].apply(lambda x: x["lat"])
parking_voiture["lon"] = parking_voiture["geo_point_2d"].apply(lambda x: x["lon"])
parking_voiture = parking_voiture.dropna(subset=["lat", "lon"])

print(f"✅ {len(parking_voiture):,} emplacements voiture chargés")

# ── BallTree haversine ─────────────────────────────────────────────────────────
parking_coords_rad = np.radians(parking_voiture[["lat", "lon"]].values)
tree_parking = BallTree(parking_coords_rad, metric="haversine")

# Coordonnées des appartements (train + test) en radians
def add_parking_features(X, tree, radius_m=500):
    coords_rad = np.radians(X[["latitude", "longitude"]].values)
    # Distance au parking le plus proche
    dist_rad, _ = tree.query(coords_rad, k=1)
    dist_m = dist_rad[:, 0] * EARTH_RADIUS_M
    # Nombre de parkings dans 500m
    radius_rad = radius_m / EARTH_RADIUS_M
    counts = tree.query_radius(coords_rad, r=radius_rad, count_only=True)
    return dist_m, counts

dist_train, count_train = add_parking_features(X_train, tree_parking)
dist_test,  count_test  = add_parking_features(X_test,  tree_parking)

X_train["distance_parking_plus_proche"] = dist_train
X_train["nb_parkings_500m"]             = count_train
X_test["distance_parking_plus_proche"]  = dist_test
X_test["nb_parkings_500m"]              = count_test

print(f"Distance moyenne au parking le plus proche : {dist_train.mean():.0f} m")
print(f"Nb moyen de parkings dans 500m : {count_train.mean():.1f}")
print(f"\n✅ Features parking ajoutées à X_train et X_test")

## 3. Modèle 1 — Régression Linéaire (baseline)

La régression linéaire suppose que le prix est une combinaison **linéaire** des features.
On utilise un `RobustScaler` dans un pipeline sklearn pour normaliser les features avant l'entraînement (le scaler est fitté uniquement sur le train set).

In [ ]:
lr_pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("model", LinearRegression()),
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

metrics_lr = compute_metrics(y_test, y_pred_lr)
print("=== Régression Linéaire ===")
print(f"MAE  : {metrics_lr['mae']:,.0f} €/m²")
print(f"RMSE : {metrics_lr['rmse']:,.0f} €/m²")
print(f"R²   : {metrics_lr['r2']:.4f}")

joblib.dump(lr_pipeline, "../models/linear_regression.pkl")
print("\n✅ Modèle sauvegardé dans models/linear_regression.pkl")

## 4. Modèle 2 — Random Forest

Le Random Forest entraîne 200 arbres de décision sur des sous-échantillons aléatoires et fait la moyenne de leurs prédictions. Pas besoin de scaler les features (les arbres sont insensibles à l'échelle).

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

metrics_rf = compute_metrics(y_test, y_pred_rf)
print("=== Random Forest ===")
print(f"MAE  : {metrics_rf['mae']:,.0f} €/m²")
print(f"RMSE : {metrics_rf['rmse']:,.0f} €/m²")
print(f"R²   : {metrics_rf['r2']:.4f}")

joblib.dump(rf_model, "../models/random_forest.pkl")
print("\n✅ Modèle sauvegardé dans models/random_forest.pkl")

## 5. Modèle 3 — Gradient Boosting (HistGradientBoostingRegressor)

Le Gradient Boosting construit les arbres **séquentiellement** : chaque arbre corrige les erreurs du précédent. `HistGradientBoostingRegressor` est l'implémentation moderne et rapide de scikit-learn, comparable à XGBoost en performance.

In [ ]:
gb_model = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
)

gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

metrics_gb = compute_metrics(y_test, y_pred_gb)
print("=== Gradient Boosting ===")
print(f"MAE  : {metrics_gb['mae']:,.0f} €/m²")
print(f"RMSE : {metrics_gb['rmse']:,.0f} €/m²")
print(f"R²   : {metrics_gb['r2']:.4f}")

joblib.dump(gb_model, "../models/gradient_boosting.pkl")
print("\n✅ Modèle sauvegardé dans models/gradient_boosting.pkl")

## 6. Comparaison des modèles

In [ ]:
results = pd.DataFrame([
    {"Modèle": "Régression Linéaire", **metrics_lr},
    {"Modèle": "Random Forest",       **metrics_rf},
    {"Modèle": "Gradient Boosting",   **metrics_gb},
]).set_index("Modèle")

results.columns = ["MAE (€/m²)", "RMSE (€/m²)", "R²"]
results = results.round({"MAE (€/m²)": 0, "RMSE (€/m²)": 0, "R²": 4})

print("=== Comparaison des modèles (test set) ===")
display(results)

# Sauvegarde pour l'app Streamlit
import os
os.makedirs("../results", exist_ok=True)
results.reset_index().to_csv("../results/model_metrics.csv", index=False)
print("\n✅ Résultats sauvegardés dans results/model_metrics.csv")

## 7. Visualisations

### 7.1 Comparaison des MAE

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

modeles = ["Régression Linéaire", "Random Forest", "Gradient Boosting"]
couleurs = ["#4C72B0", "#DD8452", "#55A868"]
maes   = [metrics_lr["mae"],  metrics_rf["mae"],  metrics_gb["mae"]]
rmses  = [metrics_lr["rmse"], metrics_rf["rmse"], metrics_gb["rmse"]]
r2s    = [metrics_lr["r2"],   metrics_rf["r2"],   metrics_gb["r2"]]

# MAE
axes[0].bar(modeles, maes, color=couleurs)
axes[0].set_title("MAE (€/m²) — moins c'est mieux", fontsize=12)
axes[0].set_ylabel("€/m²")
for i, v in enumerate(maes):
    axes[0].text(i, v + 10, f"{v:,.0f}", ha="center", fontweight="bold")

# RMSE
axes[1].bar(modeles, rmses, color=couleurs)
axes[1].set_title("RMSE (€/m²) — moins c'est mieux", fontsize=12)
axes[1].set_ylabel("€/m²")
for i, v in enumerate(rmses):
    axes[1].text(i, v + 10, f"{v:,.0f}", ha="center", fontweight="bold")

# R²
axes[2].bar(modeles, r2s, color=couleurs)
axes[2].set_title("R² — plus c'est mieux", fontsize=12)
axes[2].set_ylabel("R²")
axes[2].set_ylim(0, 1)
for i, v in enumerate(r2s):
    axes[2].text(i, v + 0.01, f"{v:.3f}", ha="center", fontweight="bold")

plt.suptitle("Comparaison des modèles — Test set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/comparaison_modeles.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Graphique sauvegardé dans plots/comparaison_modeles.png")

### 7.2 Prédictions vs Réels (Gradient Boosting — meilleur modèle)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter : réels vs prédits
sample = np.random.choice(len(y_test), size=3000, replace=False)
axes[0].scatter(y_test.iloc[sample], y_pred_gb[sample], alpha=0.3, s=10, color="#55A868")
axes[0].plot([3000, 25000], [3000, 25000], "r--", linewidth=1.5, label="Prédiction parfaite")
axes[0].set_xlabel("Prix réel (€/m²)")
axes[0].set_ylabel("Prix prédit (€/m²)")
axes[0].set_title("Gradient Boosting — Réels vs Prédits")
axes[0].legend()

# Distribution des résidus
residus = y_test.values - y_pred_gb
axes[1].hist(residus, bins=80, color="#55A868", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Résidu (€/m²)")
axes[1].set_ylabel("Fréquence")
axes[1].set_title("Distribution des résidus — Gradient Boosting")

plt.suptitle("Analyse des prédictions Gradient Boosting", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/predictions_vs_reels.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.3 Feature Importances (Gradient Boosting — permutation importance)

In [ ]:
# Permutation importance : on mélange une feature à la fois et on mesure la dégradation du R²
perm = permutation_importance(
    gb_model, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

importances = pd.Series(perm.importances_mean, index=X_train.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 8))
colors = ["#55A868" if v >= importances.quantile(0.75) else "#AEC6CF" for v in importances]
importances.plot(kind="barh", color=colors)
plt.xlabel("Importance (baisse de R² si la feature est mélangée)")
plt.title("Feature Importances — Gradient Boosting", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/feature_importances.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTop 5 features les plus importantes :")
for feat, val in importances.sort_values(ascending=False).head(5).items():
    print(f"  {feat:<40} {val:.4f}")

## 8. Amélioration des modèles

### Stratégie d'amélioration

Les modèles initiaux donnent un R² de 0.50 au mieux. Deux leviers principaux :

1. **Log-transformation de la cible** : les prix immobiliers suivent une distribution log-normale (quelques biens très chers tirent la distribution vers la droite). En prédisant `log(prix_m2)` puis en reconvertissant avec `exp()`, le modèle a une tâche plus facile.

2. **Meilleurs hyperparamètres pour le Gradient Boosting** : avec `learning_rate=0.05`, le modèle n'a pas convergé correctement. On réduit le learning rate et on augmente le nombre d'itérations.

### 8.1 Nouvelles features créées

Trois features supplémentaires ont été ajoutées dans `src/data.py` et dans ce notebook :

| Feature | Calcul | Intérêt |
|---|---|---|
| `log_surface` | `log(surface + 1)` | Capture l'effet non-linéaire de la surface sur le prix |
| `surface_par_piece` | `surface / nb_pieces` | Proxy du standing du bien |
| `arrondissement_prix_moyen` | Prix moyen de l'arrondissement (calculé sur le train set) | Feature très informative, sans fuite de données |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_train, bins=80, color="#4C72B0", edgecolor="white")
axes[0].set_title("Distribution de prix_m2 (original)")
axes[0].set_xlabel("€/m²")

axes[1].hist(y_train_log, bins=80, color="#55A868", edgecolor="white")
axes[1].set_title("Distribution de log(prix_m2)")
axes[1].set_xlabel("log(€/m²)")

plt.suptitle("Effet du log-transform sur la distribution de la cible", fontweight="bold")
plt.tight_layout()
plt.show()
print("log(prix_m2) suit une distribution beaucoup plus symétrique → meilleure pour les modèles")

### 8.2 Random Forest amélioré (avec log-transform)

In [ ]:
rf_v2 = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,          # arbres complets — meilleure expressivité
    min_samples_leaf=3,      # évite le surapprentissage sur les feuilles
    max_features=0.5,        # utilise 50% des features à chaque split
    random_state=42,
    n_jobs=-1,
)

rf_v2.fit(X_train, y_train_log)

# Reconversion : exp() pour repasser en €/m²
y_pred_rf_v2 = np.exp(rf_v2.predict(X_test))

metrics_rf_v2 = compute_metrics(y_test, y_pred_rf_v2)
print("=== Random Forest v2 (log-transform) ===")
print(f"MAE  : {metrics_rf_v2['mae']:,.0f} €/m²  (avant : {metrics_rf['mae']:,.0f})")
print(f"RMSE : {metrics_rf_v2['rmse']:,.0f} €/m²  (avant : {metrics_rf['rmse']:,.0f})")
print(f"R²   : {metrics_rf_v2['r2']:.4f}          (avant : {metrics_rf['r2']:.4f})")

joblib.dump(rf_v2, "../models/random_forest_v2.pkl")
print("\n✅ Modèle sauvegardé dans models/random_forest_v2.pkl")

### 8.3 Gradient Boosting amélioré (avec log-transform + meilleurs hyperparamètres)

In [ ]:
gb_v2 = HistGradientBoostingRegressor(
    max_iter=1000,
    learning_rate=0.02,      # plus petit = convergence plus stable
    max_depth=8,
    min_samples_leaf=20,
    l2_regularization=0.1,   # régularisation pour éviter le surapprentissage
    random_state=42,
)

gb_v2.fit(X_train, y_train_log)

# Reconversion : exp() pour repasser en €/m²
y_pred_gb_v2 = np.exp(gb_v2.predict(X_test))

metrics_gb_v2 = compute_metrics(y_test, y_pred_gb_v2)
print("=== Gradient Boosting v2 (log-transform + tuning) ===")
print(f"MAE  : {metrics_gb_v2['mae']:,.0f} €/m²  (avant : {metrics_gb['mae']:,.0f})")
print(f"RMSE : {metrics_gb_v2['rmse']:,.0f} €/m²  (avant : {metrics_gb['rmse']:,.0f})")
print(f"R²   : {metrics_gb_v2['r2']:.4f}          (avant : {metrics_gb['r2']:.4f})")

joblib.dump(gb_v2, "../models/gradient_boosting_v2.pkl")
print("\n✅ Modèle sauvegardé dans models/gradient_boosting_v2.pkl")

### 8.4 Comparaison finale — tous les modèles

In [ ]:
comparaison = pd.DataFrame([
    {"Modèle": "Régression Linéaire (v1)",    **metrics_lr},
    {"Modèle": "Random Forest (v1)",           **metrics_rf},
    {"Modèle": "Gradient Boosting (v1)",       **metrics_gb},
    {"Modèle": "Random Forest v2 (log)",       **metrics_rf_v2},
    {"Modèle": "Gradient Boosting v2 (log)",   **metrics_gb_v2},
]).set_index("Modèle")

comparaison.columns = ["MAE (€/m²)", "RMSE (€/m²)", "R²"]
comparaison = comparaison.round({"MAE (€/m²)": 0, "RMSE (€/m²)": 0, "R²": 4})

display(comparaison)

# Mise à jour du fichier de résultats
comparaison.reset_index().to_csv("../results/model_metrics.csv", index=False)
print("✅ results/model_metrics.csv mis à jour")